<div style="border-left:6px solid #14b8a6;padding:18px 22px;background:#f7f7f7">
<p style="margin:0;color:#14b8a6;letter-spacing:2px;font-size:12px">DATA MINING E INTELIGENCIA DE NEGOCIOS · GUÍA 4 · MÉTODOS SUPERVISADOS</p>
<h1 style="margin:6px 0 4px 0;font-weight:300">Guía 4 en Google Colab: árbol de decisión para predecir quién responde</h1>
<p style="margin:0;font-size:16px;color:#444">Plantilla de trabajo del grupo. Caso Mercado Central (o la base de su organización).</p>
<p style="margin:10px 0 0 0;color:#777;font-size:13px">Profe Alejandra Otero · Universidad EAN · 2026</p>
</div>

<a href="https://colab.research.google.com/github/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/blob/main/data-mining-36993/notebooks/Guia4_Arbol_Plantilla_Colab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

Este cuaderno **reemplaza el trabajo en SPSS** de la Guía 4. Aquí van a hacer todo el recorrido de un modelo supervisado: elegir la variable que quieren predecir y las variables que la explican, partir los datos en entrenamiento (80 %) y prueba (20 %), entrenar el árbol, leer sus reglas, evaluar qué tan bien acierta con la matriz de confusión y traducir todo a decisiones para el negocio.

### Datos del grupo

| Campo | Completar |
|---|---|
| Número de grupo en Canvas | Grupo __ |
| Integrantes (nombre completo) | 1. <br>2. <br>3. <br>4. |
| Base de datos que usamos | Mercado Central / Otra (¿cuál?) |
| Herramienta de IA | Gemini en Google Colab |
| Enlace de este cuaderno compartido como lector | (peguen aquí el enlace, ver el paso final) |

## Antes de empezar: cómo se trabaja este cuaderno

**1. Guarden su propia copia.** Este cuaderno está en modo lectura. Vayan a **Archivo > Guardar una copia en Drive**. Esa copia es la del grupo: compártanla entre ustedes con el botón **Compartir** (arriba a la derecha) como **Editor** para trabajar juntos.

**2. Los datos ya vienen listos.** El Paso 0 carga la **base Mercado Central ya depurada** directamente desde GitHub, la misma de la Guía 3. Si su grupo trabaja con la base de una organización real, usen el **Paso 0B** en lugar del Paso 0.

**3. Ustedes escriben el código con ayuda de Gemini, la IA que ya viene integrada en Colab.** Es gratuita y no necesitan ninguna otra herramienta. Para abrirla, usen el botón de **Gemini** (icono de chispa) en la parte superior de Colab, o el enlace **generar** que aparece en las celdas de código vacías.

Cada paso tiene esta estructura:

| Bloque | Qué hacen ustedes |
|---|---|
| **Qué tienen que lograr** | Leerlo con calma: es lo que Gemini debe ayudarles a construir |
| **Pistas técnicas** | Nombres de funciones y librerías que les sirven para pedirle a Gemini algo preciso |
| **Cómo saber que quedó bien** | Revisar el resultado antes de seguir |
| **Celda de código vacía** | Pegar el código, ejecutarlo con el botón de reproducir (o Ctrl + Enter) y corregir hasta que funcione |
| **Respuesta del grupo** | Interpretar el resultado con sus palabras |

**4. El prompt lo escriben ustedes.** No hay prompts de ejemplo: con lo que dice cada paso tienen todo lo necesario para pedirle a Gemini algo claro. Denle contexto (que trabajan en Python, que la tabla se llama `df` y cómo se llaman las columnas), pidan una cosa a la vez y, si sale un error, copien el mensaje completo y pídanle que lo corrija.

**5. Respeten los nombres.** Algunos pasos les piden guardar resultados con un nombre exacto (`objetivo`, `X`, `y`, `X_train`, `arbol`, `umbral`...). Úsenlos tal cual: el Paso 12 los necesita para armar el resumen que van a usar en el tablero del encuentro final.

**6. Qué se entrega:** este cuaderno completo (.ipynb) y una presentación corta de **5 diapositivas** (al final del cuaderno está la estructura).

## Paso 0. Cargar la base Mercado Central (esta celda ya está lista: solo ejecútenla)

Si su grupo usa **otra base**, no ejecuten esta celda: vayan al Paso 0B.

Al ejecutarla deben ver **2.030 clientes y 41 variables**, **0 valores faltantes** y una tasa de respuesta a la última campaña de **15,4 %**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

URL_DATOS = "https://raw.githubusercontent.com/YourFavouriteDataSuperstar/Inteligencia-de-negocios-globales/main/data-mining-36993/data/mercado_central_depurada.csv"
df = pd.read_csv(URL_DATOS)

# Parámetros económicos del caso Mercado Central (vienen de la base original de Kaggle)
COSTO_CONTACTO = 3        # lo que cuesta contactar a un cliente en una campaña
INGRESO_RESPUESTA = 11    # lo que gana Mercado Central por cada cliente que responde
NOMBRE_BASE = "Mercado Central"

print(f"La base tiene {df.shape[0]:,} clientes (filas) y {df.shape[1]} variables (columnas)")
print("Valores faltantes en toda la base:", df.isna().sum().sum())
print(f"Tasa de respuesta a la última campaña: {df['respondio_ultima_campana'].mean():.1%}")
df.head()

### Diccionario de variables principales (Mercado Central)

| Variable | Qué significa | Tipo |
|---|---|---|
| `id_cliente` | Identificador del cliente (no se usa en el modelo) | Identificador |
| `respondio_ultima_campana` | **Si respondió a la última campaña (1 = sí, 0 = no). Es la variable dependiente de esta guía** | Binaria |
| `edad` | Edad del cliente en años | Numérica |
| `nivel_educativo` | Básico, Pregrado, Maestría, Doctorado | Categórica |
| `estado_civil`, `situacion_pareja`, `vive_en_pareja` | Estado civil; si vive en pareja o solo (texto y 0/1) | Categórica |
| `ingreso_anual_hogar` | Ingreso anual del hogar | Numérica |
| `ninos_en_casa`, `adolescentes_en_casa`, `hijos_total` | Número de niños, adolescentes y total de hijos en el hogar | Numérica |
| `dias_desde_ultima_compra` | Recencia: días desde la última compra (menos es más reciente) | Numérica |
| `antiguedad_dias` | Días que lleva el cliente registrado | Numérica |
| `gasto_vinos`, `gasto_frutas`, `gasto_carnes`, `gasto_pescados`, `gasto_dulces`, `gasto_productos_premium`, `gasto_total` | Gasto por categoría y total en los últimos 2 años | Numérica |
| `compras_web`, `compras_catalogo`, `compras_tienda`, `compras_total` | Número de compras por canal y en total | Numérica |
| `compras_con_descuento`, `visitas_web_mes` | Compras con descuento y visitas al sitio web en el último mes | Numérica |
| `acepto_campana_1` a `acepto_campana_5`, `campanas_aceptadas` | Si aceptó cada una de las 5 campañas **anteriores** (0/1) y cuántas aceptó | Binaria / numérica |
| `reclamo_ultimos_2_anios` | Si presentó un reclamo (0/1) | Binaria |
| `ingreso_imputado`, `anio_imputado`, `atipico_*` | Marcas de la limpieza de la Guía 1 (**no se usan en el modelo**) | Control |
| `fecha_registro` | Fecha de registro (texto; usen `antiguedad_dias` en su lugar) | Fecha |

## Paso 0B. Solo si trabajan con la base de su organización

Si su grupo viene trabajando con datos de una organización real, este es su punto de partida. **No ejecuten el Paso 0.**

**Qué tienen que lograr:**

1. Cargar su base depurada (la de la Guía 1, ya trabajada en las guías 2 y 3) en una tabla llamada `df`.
2. Confirmar cuántos registros y variables tiene y cuántos faltantes quedan.
3. Definir los parámetros de negocio de su caso, si los tienen: cuánto cuesta una acción (contactar, visitar, llamar) y cuánto se gana cuando el resultado es positivo. Si no los tienen, déjenlos en `None` y en el Paso 9 hagan la lectura con otra métrica que tenga sentido para su organización.
4. Escribir abajo, en la respuesta del grupo, un diccionario corto de las variables que van a usar.

**Pistas técnicas:** para subir un archivo, `from google.colab import files` y `files.upload()`; luego `pd.read_csv()` o `pd.read_excel()`. También pueden montar Google Drive con `from google.colab import drive` y `drive.mount("/content/drive")`.

**Importante:** si su base tiene datos personales (nombres, teléfonos, correos, documentos), quítenlos antes de trabajar y confirmen con la organización que pueden usarla con fines académicos.

En el resto del cuaderno, cada vez que lean `respondio_ultima_campana`, reemplácenla por la variable que ustedes quieren predecir.

In [ ]:
# PASO 0B (solo para grupos con base propia). Completen y ejecuten.
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from google.colab import files
# subido = files.upload()
# df = pd.read_excel("NOMBRE_DE_SU_ARCHIVO.xlsx")   # o pd.read_csv(...)
# COSTO_CONTACTO = None        # costo de la acción, si lo conocen
# INGRESO_RESPUESTA = None     # ganancia cuando el resultado es positivo, si la conocen
# NOMBRE_BASE = "Nombre de la organización"

**Respuesta del grupo (solo base propia).** ¿Qué organización es, de dónde salieron los datos, cuántos registros tienen y qué significa cada variable que van a usar?

> Escriban aquí.

## Paso 1. Pregunta de negocio y objetivos ajustados

Corresponde al **punto 1 de la guía**: con base en la retroalimentación de las guías anteriores, ajusten sus objetivos. No necesita código. Va en la **diapositiva 1**.

**Qué tienen que lograr:** dejar escrita la pregunta que va a responder el árbol y los objetivos ya corregidos.

**Cómo saber que quedó bien:** la pregunta es de predicción o clasificación (por ejemplo, *¿Qué características permiten anticipar qué clientes van a responder a la próxima campaña?*) y cada objetivo se puede verificar con una tabla o gráfica de este cuaderno.

**Respuesta del grupo.**

- **Pregunta de negocio:**
- **Objetivo general:**
- **Objetivos específicos (2 o 3):**
  1.
  2.
  3.
- **Qué cambiamos frente a las guías anteriores y por qué (según la retroalimentación):**

## Paso 2. La variable dependiente y la línea base

Corresponde al **punto 3 de la guía** (elegir la variable dependiente). Va en la **diapositiva 1**.

En un método **supervisado** el modelo aprende de casos en los que ya conocemos la respuesta. Aquí la respuesta conocida es `respondio_ultima_campana`.

**Qué tienen que lograr:**

1. Guardar el nombre de la variable dependiente en `objetivo` (un texto: `objetivo = "respondio_ultima_campana"`).
2. Mostrar cuántos clientes hay en cada clase (0 y 1) y qué porcentaje representa cada una, en una tabla y en una gráfica de barras.
3. Calcular la **línea base**: si un "modelo" dijera siempre *no responde*, ¿qué porcentaje acertaría?

**Pistas técnicas:** `value_counts()` y `value_counts(normalize=True)`, `plot(kind="bar")`.

**Cómo saber que quedó bien:** en Mercado Central, alrededor de 15,4 % responde y 84,6 % no responde. La línea base es ese 84,6 %.

In [ ]:
# PASO 2: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Por qué la línea base es importante para juzgar el modelo? Si su árbol acierta 86 %, ¿es un gran resultado o no?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 3. Las variables independientes

Corresponde al **punto 3 de la guía** (elegir las variables independientes). Va en la **diapositiva 1**.

**Qué tienen que lograr:**

1. Elegir **entre 6 y 12 variables independientes**. Usen lo que encontraron en la Guía 2: los hallazgos de su análisis exploratorio son las hipótesis que el árbol va a confirmar o desmentir.
2. Convertir las categóricas de texto (por ejemplo, `nivel_educativo`) en columnas 0/1 (variables *dummy*).
3. Guardar la tabla de variables independientes en `X` y la variable dependiente en `y`.

**Reglas para no hacer trampa:**

- No incluyan `id_cliente`, las marcas de limpieza (`atipico_*`, `*_imputado`) ni `fecha_registro`.
- No incluyan la variable dependiente dentro de `X`.
- No usen información que no existiría **antes** de lanzar la campaña (se llama *fuga de información*). Las campañas **anteriores** sí se pueden usar.
- Si incluyen `gasto_total`, piensen si también necesitan cada gasto por categoría: el árbol puede elegir cualquiera de las dos, pero el informe se lee mejor si no repiten información.

**Opcional (pero vale la pena):** agregar el **clúster de la Guía 3** como una variable más. Copien aquí el código de K-Means de su Guía 3 (mismas variables activas, mismo k y `random_state=42`) para crear la columna `cluster`, y conviértanla en dummies. Así el proyecto se cuenta como una sola historia: *quiénes son* (clústeres) y *quién va a responder* (árbol).

**Pistas técnicas:** listas de Python, `pd.get_dummies(..., dtype=int)`, `pd.concat([...], axis=1)`, `X = ...`, `y = df[objetivo]`.

**Cómo saber que quedó bien:** `X` tiene 2.030 filas y solo columnas numéricas; `y` tiene 2.030 valores 0/1.

In [ ]:
# PASO 3: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Justifiquen cada variable independiente.

| Variable | Por qué la incluimos (qué hallazgo de la Guía 2 o 3 la respalda) | ¿Esperamos que aumente o disminuya la respuesta? |
|---|---|---|
| | | |
| | | |
| | | |

## Paso 4. Entrenamiento (80 %) y prueba (20 %)

Corresponde al **punto 4 de la guía**: generar las tablas de **entrenamiento** (80 % de los casos) y **test** (20 %).

El modelo aprende **solo** con los datos de entrenamiento. Los de prueba se guardan, como un examen sorpresa, para medir qué tan bien predice casos que nunca vio.

**Qué tienen que lograr:**

1. Partir `X` y `y` en `X_train`, `X_test`, `y_train`, `y_test`, con 20 % para prueba.
2. Que la partición sea **estratificada** (la misma proporción de clientes que responden en las dos tablas) y **reproducible** (`random_state=42`, así todos los grupos obtienen la misma partición).
3. Mostrar cuántos registros quedaron en cada tabla y la tasa de respuesta en cada una.

**Pistas técnicas:** `train_test_split` de `sklearn.model_selection` con `test_size=0.2`, `stratify=y`, `random_state=42`.

**Cómo saber que quedó bien:** en Mercado Central quedan **1.624 clientes de entrenamiento y 406 de prueba**, con una tasa de respuesta cercana a 15 % en las dos.

In [ ]:
# PASO 4: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Por qué no se evalúa el modelo con los mismos datos con los que aprendió? ¿Para qué sirve estratificar?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 5. Entrenar el árbol de decisión

Corresponde a los **puntos 5 y 6 de la guía** y al criterio de rúbrica *Ejecución del árbol de decisión*.

**Qué tienen que lograr:**

1. Entrenar un árbol de clasificación con los datos de entrenamiento y guardarlo en `arbol`.
2. Controlar su tamaño para que se pueda leer y no memorice los datos: profundidad máxima de **3 o 4 niveles** y un **mínimo de clientes por hoja** (por ejemplo, 30).
3. Comparar el porcentaje de acierto (*exactitud*) en entrenamiento y en prueba.
4. Probar al menos **dos configuraciones** (por ejemplo, profundidad 3 y 4) y quedarse con una, justificando por qué.

**Sobre el tipo de árbol:** el video del AVE usa CHAID en SPSS. En Python, `DecisionTreeClassifier` usa el algoritmo **CART**: hace cortes en dos ramas y elige cada corte con el índice de **Gini** (o con *entropía*, si lo cambian). Funciona con variables numéricas y con dummies, y no necesita estandarizar como el K-Means.

**Pistas técnicas:** `DecisionTreeClassifier` de `sklearn.tree` con `max_depth`, `min_samples_leaf`, `random_state=42`; métodos `.fit()` y `.score()`.

**Cómo saber que quedó bien:** la exactitud en prueba queda en el mismo orden que en entrenamiento (si en entrenamiento es mucho más alta, el árbol está memorizando: se llama *sobreajuste*).

In [ ]:
# PASO 5: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.**

| Configuración | Exactitud entrenamiento | Exactitud prueba | ¿Hay sobreajuste? |
|---|---|---|---|
| | | | |
| | | | |

- **Configuración elegida y por qué:**
- **Tipo de árbol y criterio de partición (en sus palabras):**

## Paso 6. Dibujar el árbol

Corresponde al **punto 6 de la guía**: *presentar y explicar el árbol de decisión generado*. Va en la **diapositiva 2**.

**Qué tienen que lograr:**

1. Una gráfica del árbol completo, legible, con el nombre de las variables y las clases con nombre (*No responde* y *Responde*).
2. Guardarla como imagen PNG para la presentación.

**Pistas técnicas:** `plot_tree` de `sklearn.tree` con `feature_names`, `class_names`, `filled=True`, `proportion=True`, `rounded=True`; `plt.figure(figsize=(...))` grande para que se lea; `plt.savefig("arbol.png", dpi=200, bbox_inches="tight")`.

**Cómo saber que quedó bien:** en cada caja se lee la condición del corte, cuántos clientes llegan y qué proporción responde.

In [ ]:
# PASO 6: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Lean el árbol de arriba hacia abajo. ¿Cuál es la primera variable que usa para separar a los clientes y qué significa eso para Mercado Central?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 7. Las reglas de partición en lenguaje de negocio

Corresponde al **punto 7 de la guía**: *explicar las reglas de partición que permiten la predicción de nuevos datos*. Va en la **diapositiva 3**.

Cada camino desde la raíz hasta una hoja es una **regla** del tipo *SI ... Y ... ENTONCES ...*.

**Qué tienen que lograr:**

1. Mostrar las reglas del árbol en texto.
2. Construir una tabla con las hojas: la condición completa, cuántos clientes de entrenamiento caen ahí y qué porcentaje de ellos respondió.
3. Ordenarla de mayor a menor tasa de respuesta y elegir las **3 reglas más útiles** para el negocio.

**Pistas técnicas:** `export_text` de `sklearn.tree` con `feature_names`; `arbol.apply(X_train)` devuelve la hoja de cada cliente, y con `groupby` sobre esa hoja calculan el tamaño y la tasa de respuesta.

**Cómo saber que quedó bien:** las tasas por hoja van desde muy bajas hasta muy altas, y las hojas suman los 1.624 clientes de entrenamiento.

In [ ]:
# PASO 7: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** Traduzcan las 3 reglas más útiles a lenguaje de negocio (sin nombres de columnas ni símbolos).

| Regla en lenguaje de negocio | Clientes | % que respondió | ¿Supera el punto de equilibrio? |
|---|---|---|---|
| *Ejemplo de forma: "Clientes que ya aceptaron dos o más campañas y viven solos"* | | | |
| | | | |
| | | | |

## Paso 8. Matriz de confusión y bondad de ajuste

Corresponde al **punto 8 de la guía**: *sobre la tabla test, generar una matriz de confusión y analizar la bondad de ajuste del modelo*. Va en la **diapositiva 4**.

**Qué tienen que lograr:**

1. Predecir con el árbol sobre `X_test` y guardar las predicciones en `y_pred`.
2. Mostrar la **matriz de confusión** (como tabla y como gráfica) y guardarla en PNG.
3. Calcular estas métricas y explicar cada una en una frase:

| Métrica | Pregunta que responde |
|---|---|
| **Exactitud** | De todos los clientes, ¿a cuántos acertó? |
| **Precisión** | De los que el modelo dijo que *sí responden*, ¿cuántos respondieron de verdad? |
| **Sensibilidad** (*recall*) | De los que *sí respondieron*, ¿a cuántos encontró el modelo? |

4. Comparar la exactitud con la **línea base** del Paso 2.

**Pistas técnicas:** `.predict()`, `confusion_matrix` y `ConfusionMatrixDisplay` de `sklearn.metrics`, `accuracy_score`, `precision_score`, `recall_score`, `classification_report`.

**Cómo saber que quedó bien:** la matriz suma los 406 clientes de prueba. Probablemente la exactitud se parezca mucho a la línea base y la sensibilidad sea baja: eso **no** es un error, es justo lo que hay que analizar.

In [ ]:
# PASO 8: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Qué tan confiable es el modelo? ¿Qué error es más costoso para Mercado Central: contactar a alguien que no iba a responder (falso positivo) o dejar de contactar a alguien que sí iba a responder (falso negativo)? ¿Por qué?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 9. Del modelo a la plata: el punto de equilibrio

Este paso conecta la matriz de confusión con la decisión del negocio. Es lo que la **junta directiva** (rol de Finanzas) les va a preguntar en el encuentro final. Va en la **diapositiva 4**.

En Mercado Central contactar a un cliente cuesta **3** y cada cliente que responde deja **11**. Entonces una campaña solo es rentable si responde **al menos 3 / 11 = 27,3 %** de los contactados. Ese es el **punto de equilibrio**.

Por defecto, el árbol dice *responde* solo cuando la probabilidad es mayor a 50 %. Pero para el negocio basta con que supere 27,3 %.

**Qué tienen que lograr:**

1. Calcular el punto de equilibrio (`COSTO_CONTACTO / INGRESO_RESPUESTA`) y guardarlo en `umbral`.
2. Obtener la probabilidad de responder que el árbol le da a cada cliente de prueba y clasificar como *contactar* a quienes superen el `umbral`.
3. Con esa nueva regla, mostrar la nueva matriz de confusión y calcular:
   - el **resultado por contacto** = INGRESO_RESPUESTA × precisión − COSTO_CONTACTO;
   - el **resultado total** de la campaña = INGRESO_RESPUESTA × verdaderos positivos − COSTO_CONTACTO × clientes contactados.
4. Comparar tres estrategias: **contactar a todos**, contactar a quienes el árbol clasifica con el umbral de 50 % y contactar a quienes superan el **umbral de negocio**.

**Pistas técnicas:** `arbol.predict_proba(X_test)[:, 1]`, comparación `>= umbral`, `astype(int)`.

**Cómo saber que quedó bien:** contactar a todos da un resultado **negativo** (alrededor de −1,3 por contacto). Si su árbol está bien construido, las otras dos estrategias dan resultados **positivos**. Fíjense que el umbral de 50 % puede ganar más *por contacto*, pero el umbral de negocio suele ganar más *en total*, porque encuentra a más clientes que sí responden. Esa es justo la discusión que deben resolver.

*Grupos con base propia:* si no tienen costos e ingresos, definan con su organización qué umbral tendría sentido (por ejemplo, la capacidad del equipo comercial) y expliquen su criterio.

In [ ]:
# PASO 9: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.**

| Estrategia | Clientes contactados (de 406) | Tasa de respuesta (precisión) | Resultado por contacto | Resultado total |
|---|---|---|---|---|
| Contactar a todos | 406 | | | |
| Umbral de 50 % (árbol por defecto) | | | | |
| Umbral de punto de equilibrio | | | | |

**¿Qué estrategia le recomiendan a Mercado Central y por qué?**

## Paso 10. ¿Qué variables pesan más?

**Qué tienen que lograr:**

1. Mostrar la **importancia de cada variable** en el árbol, ordenada de mayor a menor, en una gráfica de barras horizontales.
2. Compararla con las hipótesis del Paso 3: ¿se confirmó lo que esperaban de la Guía 2?

**Pistas técnicas:** atributo `arbol.feature_importances_`, `pd.Series(..., index=X.columns).sort_values()`, `plot(kind="barh")`. Un color de acento para las 3 más importantes y gris para el resto; título con la conclusión.

**Cómo saber que quedó bien:** las importancias suman 1 y varias variables quedan en 0 (el árbol no las usó).

In [ ]:
# PASO 10: escriban aquí (o generen con Gemini) el código de este paso y ejecútenlo.
# Pueden agregar más celdas de código debajo si lo necesitan (botón + Código).

**Respuesta del grupo.** ¿Qué hipótesis del Paso 3 se confirmaron y cuáles no? ¿Hay alguna variable que esperaban importante y el árbol no usó?

> Escriban aquí su respuesta con sus propias palabras (doble clic para editar). Toda afirmación debe salir de una tabla o gráfica de este cuaderno.

## Paso 11. Usos del modelo y decisiones que apoya

Corresponde al **punto 9 de la guía** y al criterio de rúbrica *Usos de los árboles de decisión y acciones recomendadas para el negocio*. Va en la **diapositiva 5**.

**Respuesta del grupo.**

1. **Respuesta a la pregunta de negocio del Paso 1:**
2. **Decisión que recomendamos** (qué hacer, con qué clientes, por qué canal o con qué mensaje):
3. **Cómo se usaría el modelo con clientes nuevos** (en la próxima campaña):
4. **Riesgo de equivocarnos** (qué pasa con los falsos positivos y los falsos negativos):
5. **Cómo mediríamos si funcionó** (indicador y meta):
6. **Limitaciones del modelo:**

## Paso 12. Resumen para el tablero del encuentro final (esta celda ya está lista: solo ejecútenla)

En el **encuentro final** (sábado 14 de noviembre) van a construir con IA un **tablero interactivo en HTML**. Esta celda junta en un archivo pequeño (`resumen_tablero.json`) los números clave de este cuaderno: la línea base, el tamaño de la partición, las métricas, la matriz de confusión, el resultado económico, las reglas y la importancia de las variables. Si crearon la columna `cluster`, también incluye el tamaño y la tasa de respuesta de cada clúster.

Al ejecutarla se descarga el archivo. **Guárdenlo**: lo van a pegar junto con el prompt del tablero.

Si la celda muestra un error, revisen que usaron los nombres exactos (`objetivo`, `X`, `y`, `X_train`, `X_test`, `y_train`, `y_test`, `arbol`, `umbral`).

In [ ]:
import json
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from sklearn.tree import _tree

def _num(v):
    return None if v is None else round(float(v), 4)

def _reglas(modelo, nombres, Xtr, ytr):
    t = modelo.tree_
    hojas = modelo.apply(Xtr)
    salida = []
    def recorrer(nodo, condiciones):
        if t.feature[nodo] != _tree.TREE_UNDEFINED:
            var = nombres[t.feature[nodo]]
            corte = round(float(t.threshold[nodo]), 2)
            recorrer(t.children_left[nodo], condiciones + [f"{var} <= {corte}"])
            recorrer(t.children_right[nodo], condiciones + [f"{var} > {corte}"])
        else:
            mascara = hojas == nodo
            n = int(mascara.sum())
            if n > 0:
                salida.append({"regla": " Y ".join(condiciones) if condiciones else "todos",
                               "clientes": n,
                               "tasa_respuesta": _num(np.asarray(ytr)[mascara].mean())})
    recorrer(0, [])
    return sorted(salida, key=lambda r: r["tasa_respuesta"], reverse=True)

pred_50 = arbol.predict(X_test)
prob = arbol.predict_proba(X_test)[:, 1]
u = umbral if "umbral" in globals() and umbral is not None else 0.5
pred_u = (prob >= u).astype(int)

def _bloque(pred):
    tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()
    prec = precision_score(y_test, pred, zero_division=0)
    b = {"matriz": {"verdaderos_negativos": int(tn), "falsos_positivos": int(fp),
                    "falsos_negativos": int(fn), "verdaderos_positivos": int(tp)},
         "exactitud": _num(accuracy_score(y_test, pred)), "precision": _num(prec),
         "sensibilidad": _num(recall_score(y_test, pred, zero_division=0)),
         "clientes_contactados": int(tp + fp)}
    if globals().get("COSTO_CONTACTO") and globals().get("INGRESO_RESPUESTA"):
        b["resultado_por_contacto"] = _num(INGRESO_RESPUESTA * prec - COSTO_CONTACTO)
        b["resultado_total_prueba"] = _num(INGRESO_RESPUESTA * tp - COSTO_CONTACTO * (tp + fp))
    return b

tasa = float(np.mean(y))
resumen = {
    "base": globals().get("NOMBRE_BASE", "Base del grupo"),
    "variable_dependiente": objetivo,
    "registros": int(len(y)),
    "tasa_positivos": _num(tasa),
    "linea_base_exactitud": _num(max(tasa, 1 - tasa)),
    "particion": {"entrenamiento": int(len(y_train)), "prueba": int(len(y_test))},
    "arbol": {"profundidad": int(arbol.get_depth()), "hojas": int(arbol.get_n_leaves()),
              "criterio": arbol.criterion,
              "exactitud_entrenamiento": _num(arbol.score(X_train, y_train)),
              "exactitud_prueba": _num(arbol.score(X_test, y_test))},
    "umbral_50": _bloque(pred_50),
    "umbral_negocio": dict(_bloque(pred_u), umbral=_num(u)),
    "reglas": _reglas(arbol, list(X.columns), X_train, y_train),
    "importancia_variables": {k: _num(v) for k, v in
        sorted(zip(X.columns, arbol.feature_importances_), key=lambda x: -x[1]) if v > 0},
}
if globals().get("COSTO_CONTACTO") and globals().get("INGRESO_RESPUESTA"):
    resumen["economia"] = {"costo_contacto": COSTO_CONTACTO, "ingreso_respuesta": INGRESO_RESPUESTA,
                           "punto_equilibrio": _num(COSTO_CONTACTO / INGRESO_RESPUESTA),
                           "resultado_contactar_a_todos": _num(INGRESO_RESPUESTA * tasa - COSTO_CONTACTO),
                           "resultado_total_contactar_a_todos_prueba": _num(INGRESO_RESPUESTA * int(np.sum(y_test)) - COSTO_CONTACTO * len(y_test))}
if "cluster" in df.columns:
    g = df.groupby("cluster")[objetivo].agg(["size", "mean"])
    resumen["clusteres"] = [{"cluster": str(i), "clientes": int(r["size"]),
                             "porcentaje": _num(r["size"] / len(df)), "tasa_respuesta": _num(r["mean"])}
                            for i, r in g.iterrows()]

with open("resumen_tablero.json", "w", encoding="utf-8") as f:
    json.dump(resumen, f, ensure_ascii=False, indent=2)
print(json.dumps(resumen, ensure_ascii=False, indent=2)[:1500], "...")
try:
    from google.colab import files
    files.download("resumen_tablero.json")
except Exception:
    print("Archivo guardado como resumen_tablero.json (panel de archivos, icono de carpeta).")

## Paso 13. Bitácora de uso de Gemini

**Qué tienen que lograr:** dejar constancia de cómo trabajaron con Gemini. No se califica cuánto la usaron, sino que la hayan usado con criterio.

**Respuesta del grupo.**

| Paso | Prompt principal que le escribimos a Gemini (cópienlo tal cual) | ¿Funcionó a la primera? Qué tuvimos que corregir o preguntar |
|---|---|---|
| 4 | | |
| 5 | | |
| 7 | | |
| 9 | | |

**Lo que aprendimos al pedirle código a Gemini (2 o 3 líneas):**

## La presentación: 5 diapositivas

El AVE pide un informe en PowerPoint. Con este cuaderno resuelto, la presentación es corta: **5 diapositivas**, una idea por diapositiva, con el título contando la conclusión (no *"Matriz de confusión"*, sino *"El árbol encuentra a la mitad de los clientes que responden"*).

| # | Diapositiva | Qué lleva | Sale del paso |
|---|---|---|---|
| 1 | **Qué queremos predecir y con qué** | Pregunta de negocio, objetivos ajustados, variable dependiente y variables independientes con su justificación | 1, 2 y 3 |
| 2 | **El árbol** | La imagen del árbol, el tipo de árbol (CART, Gini) y la configuración elegida (profundidad, mínimo por hoja, partición 80/20) | 4, 5 y 6 |
| 3 | **Las reglas** | Las 3 reglas más útiles en lenguaje de negocio, con su tamaño y su tasa de respuesta | 7 |
| 4 | **¿Qué tan confiable es?** | Matriz de confusión, exactitud frente a la línea base, precisión, sensibilidad y el resultado por contacto con el umbral de negocio | 8 y 9 |
| 5 | **Qué decidir** | Decisión recomendada, a quién, cómo medirla y limitaciones | 10 y 11 |

Pueden hacerla en PowerPoint, Google Slides o Canva, y subirla en **.pptx o PDF**.

## Antes de entregar: lista de chequeo

- [ ] Ejecutamos **Entorno de ejecución > Reiniciar y ejecutar todas** y todo corre sin errores, de arriba hacia abajo.
- [ ] Todas las celdas de **Respuesta del grupo** están diligenciadas.
- [ ] Llenamos los datos del grupo al inicio (integrantes, base y herramienta de IA).
- [ ] Descargamos y guardamos `resumen_tablero.json` (Paso 12) para el encuentro final.
- [ ] Compartimos el cuaderno: **Compartir > Acceso general > Cualquier persona con el enlace > Lector**, y pegamos el enlace en la tabla del inicio.
- [ ] Descargamos el cuaderno: **Archivo > Descargar > Descargar .ipynb**.
- [ ] Hicimos la presentación de **5 diapositivas**.

### Qué se sube a Canvas (tarea *Guía 4*, una sola entrega por grupo)

1. **El cuaderno de Colab** (archivo `.ipynb` con todas las celdas ejecutadas y el enlace compartido escrito al inicio).
2. **La presentación** de 5 diapositivas (.pptx o PDF).

Usen la opción **Cargar archivo** y adjunten los dos archivos en la misma entrega.

<p style="color:#777;font-size:13px">Dudas: en la tutoría del jueves 5 de noviembre a las 6:00 p. m. trabajamos este cuaderno en vivo, paso por paso. También pueden escribirme por el Inbox de Canvas. Profe Alejandra Otero.</p>